# 📊 IEEE-CIS Fraud Detection — Exploratory Data Analysis

> **Input:** `data/processed_train.csv` — produced by `01_Preprocessing.ipynb`  
> **Outputs:** Dark-mode figures → `reports/figures/`  |  Text summary → `reports/eda_summary.txt`

### Analysis Sections

| # | Topic | Key Insight |
|---|-------|-------------|
| 1 | Dataset overview & null audit | Confirms preprocessing success |
| 2 | Class imbalance | Drives metric choice (AUC-PR vs Accuracy) |
| 3 | Transaction amount distributions | Exposes low-value card-testing fraud |
| 4 | Temporal patterns (hour / day) | Night-time fraud spikes |
| 5 | Correlation heatmap | Fast feature importance proxy |
| 6 | Top-4 feature KDEs | How features separate fraud from legit |
| 7 | Fraud rate by amount bucket | Dual-axis volume vs risk view |

## ⚙️ 0 — Setup: Imports, Paths & Global Style

We configure a consistent **dark-mode** colour palette across all figures.
This makes visualisations both visually striking and easy to distinguish fraud (red) from legitimate (blue).

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
%matplotlib inline

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PATH   = os.path.join(BASE_DIR, 'data', 'processed_train.csv')
FIG_DIR     = os.path.join(BASE_DIR, 'reports', 'figures')
REPORT_PATH = os.path.join(BASE_DIR, 'reports', 'eda_summary.txt')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)

# ── Global colour palette ─────────────────────────────────────────────────────
BG     = '#0F1117'  # background
FRAUD  = '#E74C3C'  # red  — fraudulent transactions
LEGIT  = '#4A90D9'  # blue — legitimate transactions
ACCENT = '#F39C12'  # amber — averages / highlights
TEXT   = '#EAEAEA'  # light text
GRID   = '#2A2D3E'  # subtle grid lines

plt.rcParams.update({
    'figure.facecolor': BG,   'axes.facecolor':   BG,
    'axes.edgecolor':   GRID, 'axes.labelcolor':  TEXT,
    'axes.titlecolor':  TEXT, 'xtick.color':      TEXT,
    'ytick.color':      TEXT, 'grid.color':       GRID,
    'text.color':       TEXT, 'legend.facecolor': '#1A1D2E',
    'legend.edgecolor': GRID, 'font.size':        11,
    'axes.titlesize':   14,   'axes.labelsize':   12,
})

def savefig(fig, name):
    """Save figure to reports/figures/ and display inline."""
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, bbox_inches='tight', dpi=150, facecolor=BG)
    print(f'  Figure saved → reports/figures/{name}')
    plt.show()

print('Setup complete.')

## 📂 Load Processed Dataset

In [ ]:
print('Loading processed dataset ...')
df = pd.read_csv(DATA_PATH)
print(f'Loaded {df.shape[0]:,} rows × {df.shape[1]:,} columns')
df.head(3)

## 🗂️ Section 1 — Dataset Overview & Null Audit

### What
Print basic shape, dtype breakdown, memory footprint, and any residual missing values.

### Why
Confirms the preprocessing pipeline achieved its goals: nulls should be minimal (only columns
below the 85 % threshold were kept), dtypes should be compact, and no object columns should remain
after frequency encoding. Any surprises here indicate a preprocessing bug.

In [ ]:
print(f'Shape          : {df.shape}')
print(f'Memory         : {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print(f'Numeric cols   : {df.select_dtypes(include="number").shape[1]}')
print(f'Object cols    : {df.select_dtypes(include="object").shape[1]}')

nan_summary = df.isnull().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False).head(15)

if nan_summary.empty:
    print('\n✅  No missing values remain after preprocessing!')
else:
    print(f'\nResidual NaN columns ({len(nan_summary)} shown):')
    for col, cnt in nan_summary.items():
        print(f'  {col:<35} {cnt:>8,}  ({100*cnt/len(df):.1f}%)')

df.dtypes.value_counts()

## ⚖️ Section 2 — Class Imbalance Analysis

### What
Bar chart + pie chart showing the ratio of legitimate to fraudulent transactions.

### Why
Fraud datasets are almost always severely imbalanced (~3 % fraud). This imbalance drives
critical downstream decisions:
- **Accuracy is misleading** — always predicting 'legitimate' achieves ~97 % accuracy but catches 0 % of fraud.
- **Better metrics:** AUC-PR (area under precision-recall curve) or F1-score, both robust to imbalance.
- **Training strategies:** class-weight balancing or SMOTE oversampling of the minority class.

In [ ]:
counts = df['isFraud'].value_counts()
pct_f  = 100 * counts[1] / len(df)
pct_l  = 100 * counts[0] / len(df)

print(f'Legitimate : {counts[0]:,}  ({pct_l:.2f} %)')
print(f'Fraudulent : {counts[1]:,}  ({pct_f:.2f} %)')
print(f'Imbalance  : {counts[0]/counts[1]:.1f}:1  (legit:fraud)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution — Fraudulent vs Legitimate', fontsize=16, y=1.01)

# Bar chart
bars = ax1.bar(['Legitimate', 'Fraudulent'], [counts[0], counts[1]],
               color=[LEGIT, FRAUD], width=0.5, edgecolor='none')
ax1.set_title('Transaction Counts'); ax1.set_ylabel('Count')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.set_ylim(0, counts[0] * 1.2)
for bar, cnt, pct in zip(bars, [counts[0], counts[1]], [pct_l, pct_f]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
             f'{cnt:,}\n({pct:.1f}%)', ha='center', color=TEXT, fontweight='bold')

# Pie chart
wedges, texts, autos = ax2.pie(
    [counts[0], counts[1]], labels=['Legitimate', 'Fraudulent'],
    colors=[LEGIT, FRAUD], autopct='%1.2f%%', startangle=140,
    wedgeprops=dict(edgecolor=BG, linewidth=2), pctdistance=0.75
)
for t in autos:
    t.set_color(TEXT); t.set_fontsize(12)
ax2.set_title('Proportion')

fig.tight_layout()
savefig(fig, '02_class_imbalance.png')

## 💵 Section 3 — Transaction Amount Distributions

### What
KDE (density) and log-scale box plots of `TransactionAmt` split by fraud label, plus descriptive statistics.

### Why
Fraudsters exhibit characteristic amount patterns:
- **Low-value card testing** — tiny transactions to verify a stolen card is live before escalating.
- **High-value automated fraud** — bots drain accounts with large, quick transactions.
- **Log scale** is essential because transaction amounts span several orders of magnitude
  (cents to thousands of dollars), compressing the distribution into an unreadable spike on a linear axis.

In [ ]:
legit_amt = df.loc[df['isFraud'] == 0, 'TransactionAmt'].dropna()
fraud_amt = df.loc[df['isFraud'] == 1, 'TransactionAmt'].dropna()

for label, s in [('Legitimate', legit_amt), ('Fraudulent', fraud_amt)]:
    print(f'{label}:  mean={s.mean():.2f}  median={s.median():.2f}  '
          f'std={s.std():.2f}  max={s.max():.2f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transaction Amount Distribution by Fraud Label', fontsize=16)

p99 = df['TransactionAmt'].quantile(0.99)
sns.kdeplot(legit_amt.clip(upper=p99), ax=ax1, color=LEGIT,
            label='Legitimate', fill=True, alpha=0.35, linewidth=2)
sns.kdeplot(fraud_amt.clip(upper=p99), ax=ax1, color=FRAUD,
            label='Fraudulent', fill=True, alpha=0.50, linewidth=2)
ax1.set_xlabel('TransactionAmt (clipped at 99th pct)')
ax1.set_ylabel('Density'); ax1.set_title('KDE — Linear Scale'); ax1.legend()

plot_df = pd.DataFrame({
    'Amount': pd.concat([legit_amt, fraud_amt]),
    'Label': ['Legitimate'] * len(legit_amt) + ['Fraudulent'] * len(fraud_amt),
})
sns.boxplot(data=plot_df, x='Label', y='Amount', palette=[LEGIT, FRAUD],
            ax=ax2, linewidth=1.5,
            flierprops=dict(marker='.', color=ACCENT, markersize=2, alpha=0.3))
ax2.set_yscale('log')
ax2.set_title('Box Plot — Log Scale'); ax2.set_xlabel(''); ax2.set_ylabel('TransactionAmt (log)')

fig.tight_layout()
savefig(fig, '03_transaction_amount.png')

## ⏱️ Section 4 — Temporal Fraud Patterns

### What
Bar charts of fraud rate grouped by `Transaction_Hour` and `Transaction_Day` — the two features
we engineered in the preprocessing step from the raw `TransactionDT` column.

### Why
Fraud is *not* uniformly distributed over time; it follows predictable rhythms:
- **Hour of day:** Fraudulent bots often run at off-peak hours (late night / early morning) when
  real-time fraud monitoring has reduced coverage.
- **Day of week:** Some fraud patterns peak on specific days depending on payment processing cycles.

This plot simultaneously **validates our feature engineering** and **surfaces a directly usable insight**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Temporal Fraud Patterns', fontsize=16)

for ax, col, title, xlabel in zip(
    axes,
    ['Transaction_Hour', 'Transaction_Day'],
    ['Fraud Rate by Hour of Day',   'Fraud Rate by Day of Week'],
    ['Hour (0 = midnight)',          'Day (0 = Mon,  6 = Sun)'],
):
    if col not in df.columns:
        ax.set_visible(False)
        print(f'  Column {col!r} not found — skipping.')
        continue
    grp = df.groupby(col)['isFraud'].mean().reset_index()
    grp.columns = [col, 'fraud_rate']
    ax.bar(grp[col], grp['fraud_rate'] * 100, color=FRAUD, edgecolor='none', alpha=0.85)
    ax.axhline(df['isFraud'].mean() * 100, color=ACCENT, linestyle='--',
               linewidth=1.5, label='Overall avg')
    ax.set_xlabel(xlabel); ax.set_ylabel('Fraud Rate (%)')
    ax.set_title(title); ax.legend()

fig.tight_layout()
savefig(fig, '04_temporal_patterns.png')

## 🔥 Section 5 — Correlation Heatmap (Top 25 Features vs `isFraud`)

### What
Compute Pearson |ρ| between every numeric feature and `isFraud`, select the top 25,
and render a lower-triangular correlation heatmap.

### Why
- Identifies the most **linearly discriminative features** before any model training.
- Reveals **multicollinearity** between features (e.g. the `C`-series counter columns),
  helping prune redundant inputs and speed up training.
- Acts as a **fast, model-agnostic feature-importance proxy** — particularly useful for
  communicating findings to non-technical stakeholders.

In [ ]:
num_df = df.select_dtypes(include='number')
corr_target = num_df.corr()['isFraud'].drop('isFraud').abs().sort_values(ascending=False)
top25 = corr_target.head(25).index.tolist()

print('Top 10 features by |Pearson ρ| with isFraud:')
for feat, val in corr_target.head(10).items():
    print(f'  {feat:<35} {val:.4f}')

corr_matrix = num_df[top25 + ['isFraud']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    corr_matrix, mask=mask, ax=ax,
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.3, linecolor=BG,
    annot=(len(corr_matrix) <= 20), fmt='.2f', annot_kws={'size': 7},
    cbar_kws={'shrink': 0.75, 'label': 'Pearson ρ'},
)
ax.set_title('Correlation Matrix — Top 25 Features + isFraud', fontsize=15, pad=15)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
fig.tight_layout()
savefig(fig, '05_correlation_heatmap.png')

## 📐 Section 6 — Top Feature Distributions: Fraud vs Legitimate

### What
KDE plots for the **4 most correlated features**, showing the distribution separately for
fraudulent and legitimate transactions side by side.

### Why
Correlation tells us *that* a feature is predictive; this plot reveals *how*:
- Is fraud concentrated in the left or right tail?
- Is the distribution bimodal or uniformly shifted?
- Could a simple threshold rule catch most fraud?

These insights directly inform feature transformation decisions (log-scaling, binning, clipping)
before model training.

In [ ]:
top4 = [c for c in corr_target.head(10).index
        if c not in ('TransactionID', 'TransactionDT')][:4]
print('Top-4 features selected:', top4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
fig.suptitle('Feature Distributions — Fraud vs Legitimate', fontsize=16)

for ax, col in zip(axes, top4):
    p01, p99 = df[col].quantile([0.01, 0.99])
    legit = df.loc[df['isFraud'] == 0, col].dropna().clip(p01, p99)
    fraud = df.loc[df['isFraud'] == 1, col].dropna().clip(p01, p99)
    sns.kdeplot(legit, ax=ax, color=LEGIT, label='Legitimate', fill=True, alpha=0.30, linewidth=2)
    sns.kdeplot(fraud, ax=ax, color=FRAUD, label='Fraudulent', fill=True, alpha=0.55, linewidth=2)
    ax.set_title(col, fontsize=11); ax.set_xlabel(''); ax.legend(fontsize=9)

fig.tight_layout()
savefig(fig, '06_topfeature_distributions.png')

## 🪣 Section 7 — Fraud Rate by Transaction Amount Bucket

### What
Bin `TransactionAmt` into eight intuitive dollar ranges and overlay:
- **Blue bars** (right axis) — number of transactions in each bucket.
- **Red line** (left axis) — fraud rate within each bucket.
- **Amber dashed line** — overall average fraud rate.

### Why
Fraud risk is **not monotonic** across transaction sizes:
- **< \$10** — classic card-testing fraud (disproportionately high fraud rate).
- **\$1 K–\$5 K** — high-value automated fraud scripts targeting premium cards.
- **Mid-range \$100–\$500** — dominated by legitimate everyday purchases (low fraud rate).

This plot directly motivates `TransactionAmt` as a key feature and potentially as an
interaction term (e.g. `amt × card_type`) in the final model.

In [ ]:
bins   = [0, 10, 50, 100, 250, 500, 1000, 5000, float('inf')]
labels = ['<$10','$10-50','$50-100','$100-250','$250-500','$500-1K','$1K-5K','>$5K']

df2 = df[['TransactionAmt', 'isFraud']].dropna().copy()
df2['AmtBucket'] = pd.cut(df2['TransactionAmt'], bins=bins, labels=labels)
bucket = (
    df2.groupby('AmtBucket', observed=True)['isFraud']
    .agg(['mean', 'count'])
    .reset_index()
)
bucket.columns = ['AmtBucket', 'fraud_rate', 'n']

print('Fraud rate by amount bucket:')
for _, r in bucket.iterrows():
    print(f"  {str(r['AmtBucket']):<12}  {r['fraud_rate']*100:5.2f}%   n={int(r['n']):,}")

fig, ax1 = plt.subplots(figsize=(13, 5))
fig.suptitle('Fraud Rate & Volume by Transaction Amount Bucket', fontsize=15)
x = range(len(bucket))

ax2 = ax1.twinx()
ax2.bar(x, bucket['n'], color=LEGIT, alpha=0.20, label='# Transactions')
ax2.set_ylabel('Transaction Count', color=LEGIT)
ax2.tick_params(axis='y', colors=LEGIT)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))

ax1.plot(x, bucket['fraud_rate'] * 100, color=FRAUD, marker='o',
         linewidth=2.5, markersize=8, label='Fraud Rate %', zorder=5)
ax1.axhline(df['isFraud'].mean() * 100, color=ACCENT, linestyle='--',
            linewidth=1.2, label='Overall avg', zorder=4)
ax1.set_ylabel('Fraud Rate (%)', color=FRAUD)
ax1.tick_params(axis='y', colors=FRAUD)
ax1.set_xticks(list(x))
ax1.set_xticklabels(bucket['AmtBucket'].astype(str), rotation=20, ha='right')
ax1.set_xlabel('Transaction Amount Bucket')

l1, labs1 = ax1.get_legend_handles_labels()
l2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, labs1 + labs2, loc='upper right')

fig.tight_layout()
savefig(fig, '07_fraud_by_amount_bucket.png')
print('\n✅  All EDA sections complete!  Check reports/figures/ for the saved charts.')